# Resume Builder — Llama 3 8B QLoRA on Colab T4

This notebook trains a PEFT/LoRA adapter for `meta-llama/Meta-Llama-3-8B-Instruct` using the reviewed 120-record training split. It never consumes `holdout_eval.jsonl`, and it deliberately stops before model merge or GGUF conversion so the untouched holdout can be evaluated first.

## Manual preparation

1. In Colab select **Runtime → Change runtime type → T4 GPU**.
2. Run the cells in order.
3. Authorize Google Drive if you want resumable checkpoints.
4. Log in to Hugging Face with a **read** token belonging to the account approved for the gated model. Never paste the token into notebook source.
5. Upload only `backend/data/finetuning/training_data.jsonl`. Keep `holdout_eval.jsonl` local and untouched.

The run uses NF4 4-bit quantization, double quantization, gradient checkpointing, batch size 1, and 1024-token sequences to fit a T4 safely.

## 1. Install the training stack

Run this before importing Transformers. Colab's preinstalled PyTorch is retained.

In [ ]:
%pip install -q --upgrade "transformers>=5.17,<6" "trl>=0.29,<1" "peft>=0.21,<1" "accelerate>=1.15,<2" "bitsandbytes>=0.50,<1" "datasets>=4,<5" "huggingface_hub>=1,<2"

## 2. Verify the GPU and create a persistent workspace

In [ ]:
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select Runtime > Change runtime type > T4 GPU, then reconnect.")

gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 1024**3
USE_BF16 = bool(torch.cuda.is_bf16_supported())
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"GPU: {gpu.name}; VRAM: {vram_gib:.1f} GiB; CUDA: {torch.version.cuda}; compute dtype: {COMPUTE_DTYPE}")
if vram_gib < 14:
    raise RuntimeError(f"At least 14 GiB VRAM is required for this 8B workflow; detected {vram_gib:.1f} GiB.")

USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/resume-builder-qlora")
else:
    WORK_DIR = Path("/content/resume-builder-qlora")

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = WORK_DIR / "resume-llama3-lora"
DATASET_PATH = WORK_DIR / "training_data.jsonl"
print(f"Workspace: {WORK_DIR}")

## 3. Authenticate and prove gated-file access

The login widget stores the token in the Colab session. Use a read-only token; do not place it in code or save it to Drive.

In [ ]:
from huggingface_hub import hf_hub_download, notebook_login, whoami

BASE_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"
notebook_login()
identity = whoami()
hf_hub_download(repo_id=BASE_MODEL, filename="config.json")
print(f"Authenticated as: {identity.get('name', 'unknown')}")
print(f"Gated access verified: {BASE_MODEL}/config.json")

## 4. Upload and strictly validate the reviewed training split

Upload `training_data.jsonl` when prompted. The validator requires 120 unique records, the verified candidate name, non-empty experience, and exactly three chat turns. It refuses `holdout_eval.jsonl`.

In [ ]:
from google.colab import files
import hashlib
import json
import random
import shutil

EXPECTED_NAME = "Joji George Alungal"
EXPECTED_DATASET_SHA256 = "72baab567553c43ec815db089905d23f108978a50039cffeb18e4bfc0d64d70d"
EXPECTED_ROLES = ("system", "user", "assistant")
REQUIRED_KEYS = {"name", "title", "contact", "summary", "skills", "experience", "projects", "education"}

if not DATASET_PATH.exists():
    uploaded = files.upload()
    if "holdout_eval.jsonl" in uploaded:
        raise ValueError("Holdout upload refused. Keep holdout_eval.jsonl outside Colab training.")
    if "training_data.jsonl" not in uploaded:
        raise FileNotFoundError("Upload the file named training_data.jsonl.")
    Path("training_data.jsonl").write_bytes(uploaded["training_data.jsonl"])
    shutil.copy2("training_data.jsonl", DATASET_PATH)

dataset_bytes = DATASET_PATH.read_bytes()
dataset_sha256 = hashlib.sha256(dataset_bytes).hexdigest()
if dataset_sha256 != EXPECTED_DATASET_SHA256:
    raise ValueError(f"Dataset hash mismatch: expected {EXPECTED_DATASET_SHA256}, received {dataset_sha256}.")
records = []
seen = set()
for line_number, raw_line in enumerate(DATASET_PATH.read_text(encoding="utf-8-sig").splitlines(), 1):
    if not raw_line.strip():
        continue
    record = json.loads(raw_line)
    messages = record.get("messages")
    if not isinstance(messages, list) or tuple(m.get("role") for m in messages) != EXPECTED_ROLES:
        raise ValueError(f"Line {line_number}: expected exactly system, user, assistant turns.")
    if any(not isinstance(m.get("content"), str) or not m["content"].strip() for m in messages):
        raise ValueError(f"Line {line_number}: empty message content.")
    result = json.loads(messages[-1]["content"])
    missing = REQUIRED_KEYS - set(result)
    if missing:
        raise ValueError(f"Line {line_number}: missing output keys {sorted(missing)}.")
    if result.get("name") != EXPECTED_NAME:
        raise ValueError(f"Line {line_number}: candidate name changed.")
    if not isinstance(result.get("experience"), list) or not result["experience"]:
        raise ValueError(f"Line {line_number}: verified experience is missing.")
    digest = hashlib.sha256(json.dumps(messages, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
    if digest in seen:
        raise ValueError(f"Line {line_number}: duplicate training record.")
    seen.add(digest)
    records.append({"messages": messages})

if len(records) != 120:
    raise ValueError(f"Expected exactly 120 reviewed training records; found {len(records)}.")

random.Random(42).shuffle(records)
internal_eval_records = records[:12]
train_records = records[12:]
print(f"Validated dataset SHA-256: {dataset_sha256}")
print(f"Internal split: {len(train_records)} train / {len(internal_eval_records)} loss-monitoring eval")
print("Untouched holdout records used: 0")

## 5. Load Llama 3 in NF4 and prepare the datasets

The T4 uses FP16 compute. The base weights remain frozen and only LoRA adapter parameters are trained.

In [ ]:
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    device_map={"": 0},
    dtype=COMPUTE_DTYPE,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

def format_records(items):
    return [
        {"text": tokenizer.apply_chat_template(item["messages"], tokenize=False, add_generation_prompt=False)}
        for item in items
    ]

train_dataset = Dataset.from_list(format_records(train_records))
eval_dataset = Dataset.from_list(format_records(internal_eval_records))
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)
print(f"CUDA allocated after load: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")

## 6. Train or resume the adapter

When Drive checkpointing is enabled, rerunning this cell resumes from the newest `checkpoint-*` directory. Typical T4 runtime is tens of minutes, depending on sequence lengths and Colab load.

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    dataset_text_field="text",
    max_length=1024,
    packing=False,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=1,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    seed=42,
    data_seed=42,
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

# PEFT/AMP safety: adapter weights must be FP32 even when the frozen base uses FP16/BF16.
for parameter in trainer.model.parameters():
    if parameter.requires_grad and parameter.dtype != torch.float32:
        parameter.data = parameter.data.to(torch.float32)
trainable_dtypes = {parameter.dtype for parameter in trainer.model.parameters() if parameter.requires_grad}
trainable_count = sum(parameter.numel() for parameter in trainer.model.parameters() if parameter.requires_grad)
if trainable_dtypes != {torch.float32}:
    raise RuntimeError(f"Unsafe trainable parameter dtypes: {trainable_dtypes}")
print(f"Trainable adapter parameters: {trainable_count:,}; dtypes: {trainable_dtypes}")

checkpoints = sorted(OUTPUT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
resume_checkpoint = str(checkpoints[-1]) if checkpoints else None
print(f"Resume checkpoint: {resume_checkpoint or 'none'}")
train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
print(train_result.metrics)

## 7. Save the adapter, manifest, and portable archive

This exports only the adapter and tokenizer—not merged weights or GGUF. Download the ZIP and return it to the local project for holdout evaluation.

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version

FINAL_ADAPTER_DIR = OUTPUT_DIR / "final_adapter"
FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))

manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "base_model": BASE_MODEL,
    "base_revision": getattr(model.config, "_commit_hash", None),
    "dataset_sha256": dataset_sha256,
    "reviewed_records": len(records),
    "train_records": len(train_records),
    "internal_eval_records": len(internal_eval_records),
    "untouched_holdout_records_used": 0,
    "gpu": gpu.name,
    "vram_gib": round(vram_gib, 2),
    "training_metrics": train_result.metrics,
    "package_versions": {
        name: version(name)
        for name in ("torch", "transformers", "datasets", "accelerate", "peft", "trl", "bitsandbytes")
    },
    "training_config": {
        "epochs": 3,
        "batch_size": 1,
        "gradient_accumulation_steps": 8,
        "learning_rate": 1e-4,
        "max_length": 1024,
        "lora_r": 16,
        "lora_alpha": 32,
        "seed": 42,
    },
}
(FINAL_ADAPTER_DIR / "training_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
archive_path = shutil.make_archive(str(WORK_DIR / "resume-llama3-lora-adapter"), "zip", FINAL_ADAPTER_DIR)
print(f"Adapter: {FINAL_ADAPTER_DIR}")
print(f"Archive: {archive_path}")
print(json.dumps(manifest, indent=2))
files.download(archive_path)

## Stop here: evaluation gate

Do not merge, quantize, or register this adapter in Ollama yet. Copy `resume-llama3-lora-adapter.zip` back to the local project, verify its manifest and hashes, then evaluate it against the untouched 30-record `holdout_eval.jsonl`. Merge/GGUF export is allowed only after the adapter beats the recorded baseline without new factual errors.